# 🎙️ Chatterbox Voice Clone — CosmosGuide

Clone your own voice with [Resemble AI's Chatterbox](https://github.com/resemble-ai/chatterbox) and turn video scripts into narration.

**How to use (first time):**
1. `Runtime → Change runtime type → T4 GPU` (free tier is fine).
2. Run **Step 1**. The runtime restarts automatically once — that's expected.
3. Run **Steps 2 → 5** in order. Your voice sample and all narrations are saved in Google Drive under `MyDrive/VoiceClone/`.

**Next time:** run Step 1 (restarts), Step 2, Step 4, then go straight to Step 5. Your reference voice is already in Drive.

In [ ]:
#@title Step 1 — Install Chatterbox (runtime restarts automatically once)
import os, subprocess, sys

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ No GPU! Runtime → Change runtime type → T4 GPU"

MARKER = "/content/.chatterbox_installed"
if not os.path.exists(MARKER):
    !pip install -q chatterbox-tts pydub
    open(MARKER, "w").close()
    print("✅ Installed. Restarting runtime to load the new packages...")
    print("👉 After the restart, continue with Step 2 (do NOT re-run Step 1).")
    os.kill(os.getpid(), 9)
else:
    print("✅ Chatterbox already installed — continue with Step 2.")

In [ ]:
#@title Step 2 — Connect Google Drive & create folders
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

import os
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/VoiceClone"
else:
    BASE = "/content/VoiceClone"   # temporary: lost when the runtime ends

REF_DIR = f"{BASE}/reference"
OUT_DIR = f"{BASE}/outputs"
SCRIPTS_DIR = f"{BASE}/scripts"
for d in (REF_DIR, OUT_DIR, SCRIPTS_DIR):
    os.makedirs(d, exist_ok=True)
print("📁 Working folder:", BASE)
print("   reference/ → your voice sample")
print("   scripts/   → optional .txt scripts for batch mode (Step 6)")
print("   outputs/   → generated narration")

## Step 3 — Your reference voice

Record **10–30 seconds** of yourself speaking naturally (phone voice memo is fine):
- Quiet room, no music, no echo, one speaker only.
- Speak in the same tone/energy you want in your videos.
- Any format works: `.wav`, `.mp3`, `.m4a`, `.ogg`…

You only need to do this once — it's saved as `reference/my_voice.wav` in Drive. Re-run to replace it.

In [ ]:
#@title Step 3 — Upload & clean your voice sample (skip if already done)
TRIM_TO_SECONDS = 30  #@param {type:"integer"}

import os, subprocess
from google.colab import files

uploaded = files.upload()
if uploaded:
    src = list(uploaded.keys())[0]
    dst = f"{REF_DIR}/my_voice.wav"
    # mono, 24 kHz, trim leading/trailing silence, loudness-normalise, cap length
    af = ("silenceremove=start_periods=1:start_threshold=-45dB,"
          "areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,"
          "loudnorm=I=-20:TP=-2:LRA=11")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-af", af,
                    "-ac", "1", "-ar", "24000", "-t", str(TRIM_TO_SECONDS), dst], check=True)
    os.remove(src)
    dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                                         "-of", "csv=p=0", dst]).decode())
    print(f"✅ Saved {dst}  ({dur:.1f}s)")
    if dur < 6:
        print("⚠️ Under 6 seconds — clone quality will suffer (Turbo needs >5s). Record a longer sample.")
    from IPython.display import Audio, display
    display(Audio(dst))

In [ ]:
#@title Step 4 — Load the model
MODEL = "Multilingual (Hindi, English + 21 more)"  #@param ["Multilingual (Hindi, English + 21 more)", "English (original)", "Turbo (English, fastest)"]

import torch, os, gc
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("⚠️ Running on CPU — this will be very slow. Switch to a T4 GPU runtime.")

for _name in ("model",):
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

if MODEL.startswith("Multilingual"):
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
    MODEL_KIND = "mtl"
elif MODEL.startswith("Turbo"):
    from chatterbox.tts_turbo import ChatterboxTurboTTS
    model = ChatterboxTurboTTS.from_pretrained(device=DEVICE)
    MODEL_KIND = "turbo"
else:
    from chatterbox.tts import ChatterboxTTS
    model = ChatterboxTTS.from_pretrained(device=DEVICE)
    MODEL_KIND = "en"

REF_VOICE = f"{REF_DIR}/my_voice.wav"
assert os.path.exists(REF_VOICE), "No reference voice found — run Step 3 first."
print(f"✅ Loaded {MODEL} on {DEVICE}. Reference voice: {REF_VOICE}")

In [ ]:
#@title ⚙️ Helper functions (run once — needed by Steps 5 & 6)
import re, time, numpy as np, torch, torchaudio as ta
from pydub import AudioSegment

def split_script(text, max_chars=250):
    """Split a long script into sentence-based chunks Chatterbox handles well (~250 chars)."""
    text = re.sub(r"[ \t]+", " ", text.strip())
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []  # (chunk_text, ends_paragraph)
    for para in paragraphs:
        para = para.replace("\n", " ")
        sentences = re.split(r"(?<=[.!?।॥])\s+", para)
        cur = ""
        for s in sentences:
            # very long sentence → split on commas
            parts = [s] if len(s) <= max_chars else re.split(r"(?<=[,;:])\s+", s)
            for p in parts:
                if cur and len(cur) + 1 + len(p) > max_chars:
                    chunks.append((cur, False)); cur = p
                else:
                    cur = f"{cur} {p}".strip()
        if cur:
            chunks.append((cur, True))
    return chunks

def tts_chunk(text, language, exaggeration, cfg_weight, temperature):
    if MODEL_KIND == "mtl":
        return model.generate(text, language_id=language, audio_prompt_path=REF_VOICE,
                              exaggeration=exaggeration, cfg_weight=cfg_weight, temperature=temperature)
    if MODEL_KIND == "turbo":
        return model.generate(text, audio_prompt_path=REF_VOICE, temperature=temperature)
    return model.generate(text, audio_prompt_path=REF_VOICE,
                          exaggeration=exaggeration, cfg_weight=cfg_weight, temperature=temperature)

def narrate(script, name, language="en", exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
            seed=42, sentence_pause=0.25, paragraph_pause=0.6):
    chunks = split_script(script)
    print(f"🗣️ {len(chunks)} chunks to generate…")
    sr = model.sr
    pieces, t0 = [], time.time()
    for i, (chunk, para_end) in enumerate(chunks, 1):
        if seed:
            torch.manual_seed(seed + i)
        wav = tts_chunk(chunk, language, exaggeration, cfg_weight, temperature)
        pieces.append(wav.squeeze(0).cpu())
        pause = paragraph_pause if para_end else sentence_pause
        pieces.append(torch.zeros(int(sr * pause)))
        print(f"  [{i}/{len(chunks)}] {chunk[:70]}{'…' if len(chunk) > 70 else ''}")
    audio = torch.cat(pieces).unsqueeze(0)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    safe = re.sub(r"[^\w\-]+", "_", name).strip("_") or "narration"
    wav_path = f"{OUT_DIR}/{safe}_{stamp}.wav"
    ta.save(wav_path, audio, sr)
    mp3_path = wav_path[:-4] + ".mp3"
    AudioSegment.from_wav(wav_path).export(mp3_path, format="mp3", bitrate="192k")
    print(f"✅ Done in {time.time() - t0:.0f}s — {audio.shape[1] / sr:.1f}s of audio")
    print(f"   WAV: {wav_path}\n   MP3: {mp3_path}")
    return wav_path, mp3_path

print("✅ Helpers ready.")

In [ ]:
#@title Step 5 — Write your script & generate narration
OUTPUT_NAME = "my_video_voiceover"  #@param {type:"string"}
LANGUAGE = "en"  #@param ["en", "hi", "ar", "da", "de", "el", "es", "fi", "fr", "he", "it", "ja", "ko", "ms", "nl", "no", "pl", "pt", "ru", "sv", "sw", "tr", "zh"]
#@markdown **Expressiveness:** 0.5 = natural, 0.7+ = more dramatic (lower CFG to ~0.3 when you raise this)
EXAGGERATION = 0.5  #@param {type:"slider", min:0.25, max:2.0, step:0.05}
#@markdown **CFG / pace:** lower = slower, more deliberate; 0.5 is a good default
CFG_WEIGHT = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
TEMPERATURE = 0.8  #@param {type:"slider", min:0.05, max:2.0, step:0.05}
#@markdown **Seed:** keep the same number for a consistent delivery between runs (0 = random)
SEED = 42  #@param {type:"integer"}
DOWNLOAD_MP3 = True  #@param {type:"boolean"}
#@markdown ---
#@markdown Paste your script between the triple quotes below. Blank lines = longer paragraph pause.
#@markdown (`LANGUAGE` only applies to the Multilingual model; English/Turbo ignore it.)

SCRIPT = """
Welcome back to Cosmos Guide. Today we're going to look at what the planets have in store for you this week.

Let's begin with the Moon's transit, because it sets the emotional tone for everything that follows.
"""

wav_path, mp3_path = narrate(SCRIPT, OUTPUT_NAME, language=LANGUAGE, exaggeration=EXAGGERATION,
                             cfg_weight=CFG_WEIGHT, temperature=TEMPERATURE, seed=SEED)
from IPython.display import Audio, display
display(Audio(wav_path))
if DOWNLOAD_MP3:
    from google.colab import files
    files.download(mp3_path)

## Step 6 (optional) — Batch mode

Drop any number of `.txt` scripts into `MyDrive/VoiceClone/scripts/`, then run this cell. Each file becomes one narration in `outputs/`, named after the file. Finished scripts are moved to `scripts/done/` so they aren't generated twice.

Tip: for Hindi scripts with the Multilingual model, set `LANGUAGE = "hi"` in Step 5 and run Step 5 once (or change it here).

In [ ]:
#@title Step 6 — Generate every .txt in scripts/
BATCH_LANGUAGE = "en"  #@param ["en", "hi", "ar", "da", "de", "el", "es", "fi", "fr", "he", "it", "ja", "ko", "ms", "nl", "no", "pl", "pt", "ru", "sv", "sw", "tr", "zh"]
import glob, os, shutil
done_dir = f"{SCRIPTS_DIR}/done"
os.makedirs(done_dir, exist_ok=True)
todo = sorted(glob.glob(f"{SCRIPTS_DIR}/*.txt"))
print(f"Found {len(todo)} script(s).")
for path in todo:
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== {name} ===")
    with open(path, encoding="utf-8") as f:
        narrate(f.read(), name, language=BATCH_LANGUAGE, exaggeration=EXAGGERATION,
                cfg_weight=CFG_WEIGHT, temperature=TEMPERATURE, seed=SEED)
    shutil.move(path, f"{done_dir}/{os.path.basename(path)}")

## Tips for the best clone

| Problem | Fix |
|---|---|
| Doesn't sound like me | Re-record the sample: closer to the mic, quieter room, 15–25s, natural speech (not reading flatly). |
| Too fast / rushed | Lower **CFG_WEIGHT** to 0.3. |
| Too flat / robotic | Raise **EXAGGERATION** to 0.6–0.8 and lower CFG to ~0.3. |
| Odd word or glitch in one sentence | Change **SEED** and regenerate, or rephrase that sentence. |
| Hindi sounds accented | Use the **Multilingual** model with `LANGUAGE = "hi"`, and record your reference sample in Hindi. |
| `CUDA out of memory` | `Runtime → Restart session`, then Steps 2, 4, helpers, 5. |

Generated audio carries Resemble AI's inaudible Perth watermark (built into Chatterbox). Only clone voices you own or have permission to use.